# Module 1 — Clinical Corpus Ingestion (Colab version)

**Dataset:** `PMC-Patients.csv`, loaded from Google Drive (`ClinicalTrust/data/raw/`)

**Goal:** Load patient summaries, split into overlapping chunks, embed with Bio-ClinicalBERT (on GPU), store in a FAISS index on Drive.

**Ported from your local notebook, with two changes:**
1. Paths now point to Drive instead of a local folder, and embeddings run on the T4 GPU instead of CPU.
2. The embedding pooling step is fixed to ignore padding tokens (your local version averaged over padding too, which slightly skews embeddings for shorter texts in a batch).

Run this **after `00_setup.ipynb`** has been run at least once (so the CSV is confirmed present on Drive). This notebook installs its own packages, so it works even in a brand-new Colab session.

## Step 1 — Mount Drive & confirm the dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
CSV_PATH = os.path.join(PROJECT_ROOT, "data/raw/PMC-Patients.csv")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data/processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("CSV found:", os.path.exists(CSV_PATH))
print("CSV_PATH:", CSV_PATH)
print("PROCESSED_DIR:", PROCESSED_DIR)

Mounted at /content/drive
CSV found: True
CSV_PATH: /content/drive/MyDrive/ClinicalTrust/data/raw/PMC-Patients.csv
PROCESSED_DIR: /content/drive/MyDrive/ClinicalTrust/data/processed


## Step 2 — Install packages

`torch` is already present on Colab's GPU runtime. `transformers` and `faiss-cpu` are not, so we install them here.

In [2]:
!pip install -q transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.4 MB/s eta 0:00:00


## Step 3 — Imports, GPU check, and constants

In [3]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel
import torch
import faiss

TEXT_COLUMN = "patient"
ID_COLUMN = "patient_id"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

Using device: cuda
GPU name: Tesla T4


## Step 4 — Load a subset of the CSV

Locally was limited to 2,000 rows by 8GB RAM. On Colab's T4 GPU we can go larger — this defaults to **5,000**, in line with the 2,000–5,000 patient range agreed as a defensible subset size for the thesis (rather than the full 167,034 rows, to keep runtime realistic within Colab's session limits). Raise it later once this runs cleanly.

In [4]:
SUBSET_SIZE = 5000  # raise later once verified, up to ~5000 for the thesis-scale subset

chunks_iter = pd.read_csv(CSV_PATH, chunksize=SUBSET_SIZE)
df = next(chunks_iter)

print(df.shape)
print(df.columns.tolist())
df[[ID_COLUMN, TEXT_COLUMN, "age", "gender"]].head()

(5000, 10)
['patient_id', 'patient_uid', 'PMID', 'file_path', 'title', 'patient', 'age', 'gender', 'relevant_articles', 'similar_patients']


,patient_id,patient,age,gender
0,0,This 60-year-old male was hospitalized due to ...,"[[60.0, 'year']]",M
1,1,A 39-year-old man was hospitalized due to an i...,"[[39.0, 'year']]",M
2,2,One week after a positive COVID-19 result this...,"[[57.0, 'year']]",M
3,3,This 69-year-old male was admitted to the ICU ...,"[[69.0, 'year']]",M
4,4,This 57-year-old male was admitted to the ICU ...,"[[57.0, 'year']]",M


## Step 5 — Basic cleaning

Drop any rows with missing or very short patient text (per proposal's Risk R3 — quality filter on ingestion).

In [5]:
MIN_CHARS = 50

before = len(df)
df = df.dropna(subset=[TEXT_COLUMN])
df = df[df[TEXT_COLUMN].str.len() >= MIN_CHARS].reset_index(drop=True)
after = len(df)

print(f"Dropped {before - after} rows with missing/short text. Remaining: {after}")

Dropped 0 rows with missing/short text. Remaining: 5000


## Step 6 — Chunking (512 tokens, 50-token overlap)

This uses Bio-ClinicalBERT's own tokenizer so chunk boundaries align with what the embedding model will actually see.

In [6]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def chunk_text(text, chunk_size=512, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunks.append(tokenizer.decode(chunk_tokens))
        start += chunk_size - overlap
    return chunks

# Sanity check on one row
sample_chunks = chunk_text(df[TEXT_COLUMN].iloc[0])
print(f"Row 0 produced {len(sample_chunks)} chunk(s)")
print(sample_chunks[0][:300])

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

Row 0 produced 1 chunk(s)
this 60 - year - old male was hospitalized due to moderate ards from covid - 19 with symptoms of fever, dry cough, and dyspnea. we encountered several difficulties during physical therapy on the acute ward. first, any change of position or deep breathing triggered coughing attacks that induced oxyge


## Step 7 — Build the full chunk table (with metadata)

In [7]:
records = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    chunks = chunk_text(row[TEXT_COLUMN])
    for c_idx, chunk in enumerate(chunks):
        records.append({
            "patient_id": row[ID_COLUMN],
            "chunk_id": f"{row[ID_COLUMN]}_{c_idx}",
            "age": row.get("age", None),
            "gender": row.get("gender", None),
            "text": chunk
        })

chunks_df = pd.DataFrame(records)
print(chunks_df.shape)
chunks_df.head()

  0%|          | 0/5000 [00:00<?, ?it/s]

(10531, 5)


,patient_id,chunk_id,age,gender,text
0,0,0_0,"[[60.0, 'year']]",M,this 60 - year - old male was hospitalized due...
1,1,1_0,"[[39.0, 'year']]",M,a 39 - year - old man was hospitalized due to ...
2,2,2_0,"[[57.0, 'year']]",M,one week after a positive covid - 19 result th...
3,3,3_0,"[[69.0, 'year']]",M,this 69 - year - old male was admitted to the ...
4,4,4_0,"[[57.0, 'year']]",M,this 57 - year - old male was admitted to the ...


## Step 8 — Embed each chunk with Bio-ClinicalBERT (GPU, attention-masked mean pooling)

**Fix vs. your local version:** the local code used `outputs.last_hidden_state.mean(dim=1)`, which averages over *every* token position in a batch — including padding tokens added to shorter sequences. That slightly pollutes the embedding for anything shorter than the longest sequence in its batch. This version masks out padding before averaging, using only the real tokens. On the T4 GPU this should run through 5,000 patients' worth of chunks much faster than the CPU timing you saw locally.

In [8]:
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

def embed_texts(texts, batch_size=32):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        # Attention-masked mean pooling: exclude padding tokens from the average
        mask = inputs["attention_mask"].unsqueeze(-1)
        summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = summed / counts

        all_embeddings.append(mean_pooled.cpu().numpy())
    return np.vstack(all_embeddings)

# batch_size raised to 32 since the T4 GPU handles this comfortably (was 8 locally on CPU)
embeddings = embed_texts(chunks_df["text"].tolist(), batch_size=32)
print(embeddings.shape)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/330 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

(10531, 768)


## Step 9 — Build and save the FAISS index to Drive

Saved under `data/processed/` on Drive, so it persists across Colab sessions and Module 3 can load it directly.

In [9]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

faiss.write_index(index, os.path.join(PROCESSED_DIR, "pmc_patients.index"))
chunks_df.to_parquet(os.path.join(PROCESSED_DIR, "chunks_metadata.parquet"))

print("Saved FAISS index and metadata to:", PROCESSED_DIR)
print("Total vectors in index:", index.ntotal)

Saved FAISS index and metadata to: /content/drive/MyDrive/ClinicalTrust/data/processed
Total vectors in index: 10531


## Step 10 — Quick retrieval test

In [10]:
query = "patient with type 2 diabetes and elevated creatinine"
query_embedding = embed_texts([query], batch_size=1)

k = 5
distances, indices = index.search(query_embedding.astype("float32"), k)

for rank, idx in enumerate(indices[0]):
    print(f"Rank {rank+1} (distance={distances[0][rank]:.2f}):")
    print(chunks_df.iloc[idx]["text"][:300])
    print("---")

  0%|          | 0/1 [00:00<?, ?it/s]

Rank 1 (distance=25.09):
on to achieve renal recovery [ serum cr 95 μmol / l on day 8 of hospitalization ] with restoration of urine output, acid - base and electrolyte balance. his functional status at the time of discharge was at his baseline. the metformin was discontinued from the day of admission, and he was prescribed
---
Rank 2 (distance=25.71):
follow - up, his serum creatinine downtrended and stabilized to 2. 28 mg / dl without evidence of hyperkalemia or oliguria.
---
Rank 3 (distance=26.54):
a 73 - year - old male with a history of type 2 diabetes mellitus presented with a heel ulcer. radiograph of the left foot ( ) reveals a wedge - shaped avulsion fracture at the posterior calcaneus.
---
Rank 4 (distance=26.57):
show increased metabolic uptake. seven months after pancreas resection his hba1c increased from 6. 9 to 8. 7 %. this led us to modify his diabetes medication to a combination including insulin glargine, insulin glulisine, and metformin.
---
Rank 5 (distance=27.19):

## Next steps

1. If this runs cleanly on 5,000 rows, we can try raising `SUBSET_SIZE` further — watch Colab's session/runtime limits if you do.
2. Move to **Module 2**: scispaCy NER + UMLS CUI linking + populating Aura Neo4j graph. Remember Module 2's Neo4j connection needs `auth=(NEO4J_USERNAME, NEO4J_PASSWORD)` — not the hardcoded `"neo4j"` username — since current Aura instance uses an instance-ID-style username.
3. Module 3's retriever will load `pmc_patients.index` and `chunks_metadata.parquet` directly from `data/processed/` on Drive.
4. Save this notebook itself into `ClinicalTrust/notebooks/` on Drive so it's alongside the rest of the project.